# End of week 1 exercise

To demonstrate your familiarity with OpenAI API, and also Ollama, build a tool that takes a technical question,  
and responds with an explanation. This is a tool that you will be able to use yourself during the course!

In [35]:
# imports
from IPython.display import Markdown, display
from openai import OpenAI
import os
from dotenv import load_dotenv
import sys
import json
import io
import contextlib
import traceback
import requests
import gradio as gr

In [36]:
# constants

MODEL_GPT = 'gpt-4o-mini'
MODEL_LLAMA = 'llama3.2'
MODEL_GEMINI = 'gemini-3.1-flash-lite'
MODEL_QROQ = "openai/gpt-oss-120b"


In [37]:
# set up environment
load_dotenv(override=True)


# Load environment variables in a file called .env
api_key: str | None = os.getenv('OPENAI_API_KEY')
google_api_key: str | None = os.getenv('GOOGLE_API_KEY')
groq_api_key = os.getenv('GROQ_API_KEY')


GEMINI_BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"
OLLAMA_BASE_URL = "http://localhost:11434/v1"
GROK_BASE_URL = "https://api.x.ai/v1"


if google_api_key:
    print(f"Google API Key exists and begins {google_api_key[:8]}")
else:
    print("Google API Key not set")
    
if api_key:
    print(f"OpenAI API Key exists and begins {api_key[:2]}")
else:
    print("OpenAI API Key not set (and this is optional)")

if groq_api_key:
    print(f"Grok API Key exists and begins {groq_api_key[:4]}")
else:
    print("Grok API Key not set (and this is optional)")

openAI = OpenAI()
gemini = OpenAI(base_url=GEMINI_BASE_URL, api_key=google_api_key)
ollama = OpenAI(base_url=OLLAMA_BASE_URL, api_key='ollama')
groq = OpenAI(base_url=GROK_BASE_URL, api_key=groq_api_key)

Google API Key exists and begins AQ.Ab8RN
OpenAI API Key exists and begins sk
Grok API Key exists and begins gsk_


In [38]:
system_message = """
    You are an helpful assistant to help understand the code and the concepts behind it and suggest the best way to implement it.
    You are also an expert in the field of software development and have a deep understanding of the code and the concepts behind it.
"""

## Tools

In [ ]:
DEVELOPER_TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "execute_python_sandbox",
            "description": "Executes standard Python code in an isolated sandbox environment. Use this to verify logic, run calculations, test algorithms, or validate code snippets before presenting them to the user.",
            "parameters": {
                "type": "object",
                "properties": {
                    "code": {
                        "type": "string",
                        "description": "The complete Python code string to run. Print the output or variables you want to inspect."
                    }
                },
                "required": ["code"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "search_developer_docs",
            "description": "Searches the web for up-to-date programming language documentation, framework syntax updates, API specifications, or error resolutions.",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {
                        "type": "string",
                        "description": "The search query focusing on technical keywords, e.g., 'Gradio type messages Chatbot layout documentation'"
                    }
                },
                "required": ["query"]
            }
        }
    }
]

In [40]:
def execute_python_sandbox(code: str) -> str:
    """Safely runs python code strings and returns stdout or errors."""
    f = io.StringIO()
    # Redirect standard output so we can capture print statements
    with contextlib.redirect_stdout(f):
        try:
            # We provide a clean, isolated dictionary for variables
            exposed_globals = {}
            exec(code, exposed_globals)
            output = f.getvalue()
            return output if output.strip() else "Code executed successfully with no stdout."
        except Exception as e:
            # If the LLM generates broken code, capture the exact trace so it can auto-correct
            return f"Error occurred during execution:\n{traceback.format_exc()}"

def search_developer_docs(query: str) -> str:
    """Searches the web for up-to-date programming syntax or documentation."""
    try:
        # Example using a simple, clean DuckDuckGo HTML or JSON proxy API
        # You can substitute this with 'tavily', 'exa', or 'perplexity' keys easily
        url = f"https://duckduckgo.com{requests.utils.quote(query)}"
        headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"}
        response = requests.get(url, headers=headers, timeout=10)
        
        if response.status_code == 200:
            # Simple fallback text truncation to extract some readable snippet data
            from bs4 import BeautifulSoup
            soup = BeautifulSoup(response.text, 'html.parser')
            snippets = [result.text.strip() for result in soup.find_all('a', class_='result__snippet')[:3]]
            if snippets:
                return "\n\n".join(snippets)
        return f"No documentation snippet found for query: '{query}'"
    except Exception as e:
        return f"Failed to search documentation: {str(e)}"

In [41]:
def chat(history, model_name):
    # 1. Extracted the latest user message from the updated history list
    if not history:
        return history, gr.skip()
    
    # 2. Formatted history cleanly for the API
    formatted_history = [{"role": h["role"], "content": h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + formatted_history
    
    # 4. Swapped GEMINI_BASE_URL for the actual model string variable (e.g., GEMINI_MODEL)
    # Route the request to the client corresponding to the dropdown selection
    if model_name == MODEL_GEMINI:
        client = gemini
    elif model_name == MODEL_LLAMA:
        client = ollama
    elif model_name == MODEL_QROQ:
        client = groq
    elif model_name == MODEL_GPT:
        client = openAI
    else:
        # Fallback error handling if an unknown model is selected
        raise ValueError(f"Unknown model selected: {model_name}")

    response = client.chat.completions.create(
        model=model_name, 
        messages=messages,
        tools=DEVELOPER_TOOLS
    )

    message_obj = response.choices[0].message
    
    if message_obj.tool_calls:
        print("👉 Step 1: Model requested tools. Running execution...")

        # We must append the model's tool call intent back to the API message tree
        messages.append(message_obj)
        
        for tool_call in message_obj.tool_calls:
            tool_name = tool_call.function.name
            args = json.loads(tool_call.function.arguments)
            tool_output = ""
            
            print(f"Executing tool: {tool_name} with args: {args}")
            
            # Route to the matching python backend tool handler
            if tool_name == "execute_python_sandbox":
                tool_output = execute_python_sandbox(args.get("code", ""))
            elif tool_name == "search_developer_docs":
                tool_output = search_developer_docs(args.get("query", ""))
            else:
                tool_output = f"Error: Tool '{tool_name}' is not supported."
                
            # Append the calculated tool output back into the API messages timeline
            messages.append({
                "role": "tool",
                "tool_call_id": tool_call.id,
                "name": tool_name,
                "content": tool_output
            })

        # 5. Second API Call (Passing tool execution data back to model)
        print("👉 Step 2: Sending tool execution data back to model for final answer...")
        second_response = client.chat.completions.create(
            model=model_name,
            messages=messages
        )
        bot_message = second_response.choices[0].message.content
    else:
        # Standard conversation text generation pathway
        bot_message = message_obj.content

    # If the response happens to be completely empty, add a default fallback
    if not bot_message:
        bot_message = "I analyzed the context but couldn't produce text output."

    # 6. Smoothly return results to Gradio
    history.append({"role": "assistant", "content": bot_message})
    return history

In [ ]:
# Callbacks (along with the chat() function above)

def put_message_in_chatbot(message, history):
    return "", history + [{"role": "user", "content": message}]

# UI definition
with gr.Blocks() as ui:
    with gr.Row():
        chatbot = gr.Chatbot(height=500, type="messages")
        model_dropdown = gr.Dropdown(
            choices=[MODEL_GEMINI, MODEL_LLAMA, MODEL_QROQ, MODEL_GPT], 
            value=MODEL_GEMINI, 
            label="Select Model"
        )
    with gr.Row():
        message = gr.Textbox(label="Chat with our AI Assistant:")

    # Hooking up events to callbacks
    message.submit(
        put_message_in_chatbot, 
        inputs=[message, chatbot], 
        outputs=[message, chatbot]
    ).then(
        chat, 
        inputs=[chatbot, model_dropdown],  # FIX: Added model_dropdown here to pass it into chat()
        outputs=[chatbot]
    )

ui.launch(inbrowser=True, auth=("k", "v"))

* Running on local URL:  http://127.0.0.1:7869
* To create a public link, set `share=True` in `launch()`.


python(3931) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


👉 Step 1: Model requested tools. Running execution...
Executing tool: search_developer_docs with args: {'query': 'gradio chatbot component messages API documentation'}
👉 Step 2: Sending tool execution data back to model for final answer...
👉 Step 1: Model requested tools. Running execution...
Executing tool: execute_python_sandbox with args: {'code': '# I will simulate the logic of the provided code because\n# running `demo.launch()` in a non-interactive/server-less environment\n# will trigger an error or hang.\n\n# Simulating the function logic\ndef respond(message, chat_history):\n    chat_history.append({"role": "user", "content": message})\n    chat_history.append({"role": "assistant", "content": "Hello!"})\n    return "", chat_history\n\n# Test Case\nhistory = []\nmsg_input = "Hi there!"\nnew_msg, new_history = respond(msg_input, history)\n\nprint(f"Message returned: \'{new_msg}\'")\nprint(f"History updated: {new_history}")\n'}
👉 Step 2: Sending tool execution data back to model f